# Lesson 2 : Trace Agent

In real development, you'll need to trace the detailed process, and adjust contexts - such as, customizing instructions, setting-up appropriate tools, adding memories, etc.  
Agent Framework provides tracing and logging capabilities by using OpenTelemetry standard, and you can configure to work with a wide variety of logging platforms - such as, Aspire Dashboard, Jaeger, Prometheus, etc. (You can also output trace and logs in your console.)

In this exercise, we'll trace a agent which is built in Lesson 1, and send to Microsoft Foundry tracing.

## 1. Create and connect to an Application Insight resource

In order for Microsoft Foundry tracing, you should prepare an Application Insights resource in Microsoft Foundry.  
Before you start tracing, run the following steps.

1. Open [Azure Portal](https://portal.azure.com) and create a new Application Insights resource.
2. Open Foundry Portal (new portal), click "Operate" menu, and select "Admin" in left-side navigation.
3. Select your project, and connect to above Application Insights resource by clicking "Add connection".

## 2. Tracing your agent

Same as Lesson 1, we provision a client for the agent.  
In this exercise, however, please provide ```agent_name``` and ```agent_version``` as follows, because we connect to the existing agent in Microsoft Foundry.

> Note : Instead of setting ```agent_version```, you can also set ```use_latest_version=True``` to use the latest version.

In [1]:
from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential

credential = AzureCliCredential()
client = AzureAIClient(
    credential=credential,
    agent_name="BasicWeatherAgent",
    agent_version="1",
)

Connect to the agent.

In [2]:
# define local tools
from typing import Annotated
from pydantic import Field
from random import randint

def get_weather(
    location: Annotated[str, Field(description="天気を取得する場所")],  # "the location to get the weather for"
) -> str:
    """与えられた場所の天気を取得する。"""
    conditions = ["晴れ", "曇り", "雨", "嵐"]  # "sunny", "cloudy", "rainy", "stormy"
    return f"{location}の天気は{conditions[randint(0, 3)]}です。"  # "the weather in ... is ... ."

def get_temperature(
    location: Annotated[str, Field(description="気温を取得する場所")],  # "the location to get the temperature for"
) -> str:
    """与えられた場所の気温を取得する。"""
    return f"{location}の気温は{randint(10, 30)}°Cです。"  # "the temperature in ... is ... degrees."

In [3]:
# connect to the agent
from agent_framework import ChatAgent

agent = ChatAgent(
    chat_client=client,
    instructions="あなたは、気象情報に関する Agent です。",  # "you are an agent about weather information."
    tools=[get_weather, get_temperature])

Run agent with trace enabled.  
For Microsoft Foundry tracing, you can simply use built-in ```configure_azure_monitor()``` in client. (This method internally detects the connected Application Insights connection, invokes ```configure_azure_monitor()``` in Azure observability SDK, and built-in ```enable_instrumentation()``` in Agent Framework SDK.)

For other tracing (except for Microsoft Foundry tracing), you can use built-in ```configure_otel_providers()``` and related environments. (You can also manually configure exporters, providers, and instrumentation for OpenTelemetry tracing.)<br>
See [here](https://github.com/microsoft/agent-framework/tree/main/python/samples/getting_started/observability) for examples.

In [4]:
from IPython.display import Markdown, display
from agent_framework.observability import get_tracer
from opentelemetry.trace import SpanKind
from opentelemetry.trace.span import format_trace_id

await client.configure_azure_monitor(
    enable_live_metrics=False,
    enable_sensitive_data=True,
)

with get_tracer().start_as_current_span("Weather Agent Test", kind=SpanKind.CLIENT) as current_span:
    print(f"Trace ID: {format_trace_id(current_span.get_span_context().trace_id)}")

    result = await agent.run("大阪の天気と気温を教えてください。")  # "tell me the weather and temperature in Osaka."
    display(Markdown(result.text))

Trace ID: b027bb71024afa65a29ea44deb87d96c


大阪の天気は**曇り**です。  
大阪の気温は**13°C**です。

After a while, you can view the trace in Microsoft Foundry project.

![Trace view in Microsoft Foundry](./assets/foundry_trace.png)

You can view the collected trace logs in Application Insights or Azure Monitor.  
For example, let's see and check logs calling OpenAI Responses API invoked from Agent Framework, as follows.

1. Go to Application Insights resource in [Azure Portal](https://portal.zure.com), and select your resource.
2. Expand "Investigate" in left-side navigator and click "Agents".
3. Click item in "Tool calls" or "Models" section.
4. Select a target trace session.
5. Expand call navigation in left-side, and select one of "LLM" items or "GenAI" items. (See below picture.)

![Trace view in Application Insights](./assets/appinsight_trace.png)

> Note : To view the input text and the generated output text, set ```enable_sensitive_data=True``` in ```configure_azure_monitor()```.